# NOTEBOOK 05 – VĨNH CHI | CHƯƠNG 5: KẾT QUẢ & THẢO LUẬN

**Phụ trách:** 5.1–5.4 · GitHub Issue **#6** · Bảng kết quả Test hạn 24/11, Ch5 hoàn chỉnh 25/11/2026.

**Nhận từ Trúc Chi:** `test_predictions.csv`, `experiment_metadata.json` và thông tin ngưỡng/cấu hình chọn trên Validation.

**Bàn giao Tấn Lên:** `reports/tables/model_metrics.csv`, hình PR Curve, Top-K và bản phân tích lỗi.

[Lý thuyết Ch5](../docs/theory/05_vinh_chi_ch5.md) · [Giao thức thực nghiệm](../docs/EXPERIMENT_PROTOCOL.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root():
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "docs" / "DATA_CONTRACT.md").exists():
            return candidate
    raise FileNotFoundError("Mở notebook trong repository (hoặc thư mục notebooks/).")
ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 5.1 Kiểm tra bộ kết quả Test cố định

**Không tự huấn luyện lại**; đọc xác suất và nhãn gốc từ file dự đoán Test đã được đóng băng.

Giải thích bối cảnh: thời điểm model dự đoán, tỷ lệ lớp trễ, phân chia theo thời gian; chỉ số chính AP (quy ước PR-AUC), Precision, Recall, F1, FP, FN, TN và biểu đồ PR.

In [ ]:
import json
from sklearn.metrics import (
    average_precision_score, confusion_matrix, precision_recall_curve,
    precision_score, recall_score, f1_score
)
pred_file = PROCESSED / "test_predictions.csv"
meta_file = PROCESSED / "experiment_metadata.json"
if not pred_file.exists() or not meta_file.exists():
    raise FileNotFoundError("Thiếu dữ liệu Test/metadata của Ch4. Yêu cầu Trúc Chi bàn giao.")
pred = pd.read_csv(pred_file)
meta = json.loads(meta_file.read_text(encoding="utf-8"))
required = {"order_id","y_true","p_dummy","p_rf","p_mlp","y_pred_champion"}
assert required.issubset(pred.columns), required.difference(pred.columns)
assert pred["order_id"].is_unique
assert pred["y_true"].isin([0,1]).all()
for name in ["dummy","rf","mlp"]:
    assert pred["p_"+name].between(0,1).all()
print("Test n =",len(pred),"| Late rate =",pred.y_true.mean())
print("Model đã chọn trên Validation:",meta["champion"])
print("Không chọn threshold theo Test.")

champion = meta["champion"]

assert champion in ["dummy", "rf", "mlp"]

threshold = float(
    meta["thresholds_from_val"][champion]
)

assert 0 <= threshold <= 1


reconstructed_label = (
    pred["p_" + champion] >= threshold
).astype(int)

mismatch_count = (
    reconstructed_label != pred["y_pred_champion"].astype(int)
).sum()

print("Champion:", champion)
print("Threshold từ Validation:", threshold)
print("Số nhãn Champion không khớp:", mismatch_count)
print("Các trường trong metadata:", list(meta.keys()))

assert mismatch_count == 0, (
    "Nhãn Champion không khớp ngưỡng Validation!"
)


import joblib

MODELS_DIR = ROOT / "models"
artifact_map = {
    "dummy": MODELS_DIR / "dummy_truc_chi.joblib",
    "rf": MODELS_DIR / "ngoc_lan_random_forest.joblib",
    "mlp": MODELS_DIR / "tan_len_mlp.joblib",
}
print("=== KIỂM TRA TỆP MÔ HÌNH ===")
for name, path in artifact_map.items():
        print(f"\nMô hình: {name}")
        print("Đường dẫn:", path.resolve())
        print("Tồn tại:", path.is_file())

features = meta["features"]
X_test_check = pred[["order_id"]].merge(
    pd.read_csv(PROCESSED / "orders_model_ready.csv"),
    on="order_id",
    how="left",
)[features]

reconcile_rows = []
for name, mpath in artifact_map.items():
    if mpath.exists():
        clf = joblib.load(mpath)
        p_recalc = clf.predict_proba(X_test_check)[:, 1]
        max_diff = float(np.max(np.abs(p_recalc - pred["p_" + name].to_numpy())))
        assert np.allclose(p_recalc, pred["p_" + name].to_numpy(), atol=1e-6), f"Lệch xác suất {name}!"
        reconcile_rows.append({"model": name, "artifact": mpath.name, "max_abs_diff": max_diff, "status": "PASS"})
    else:
        reconcile_rows.append({"model": name, "artifact": mpath.name, "max_abs_diff": np.nan, "status": "SKIPPED (missing local joblib)"})

display(pd.DataFrame(reconcile_rows))

## 5.1.1 Bảng chỉ số và ma trận nhầm lẫn

**Diễn giải:** Precision gắn với cảnh báo sai, Recall gắn với đơn trễ bị bỏ sót, F1 cân bằng hai khía cạnh, AP đo độ tốt xếp hạng qua ngưỡng. Luôn báo **số lượng thực (TP/FP/FN/TN)**, không chỉ phần trăm.

Ngưỡng của mỗi mô hình phải là ngưỡng đã chọn từ Validation; Dummy là mức so sánh.

In [ ]:
actual = pred["y_true"].to_numpy().astype(int)
rows = []
for name in ["dummy", "rf", "mlp"]:
    probabilities = pred["p_" + name].to_numpy()
    threshold = float(meta["thresholds_from_val"][name])
    labels = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(actual, labels, labels=[0, 1]).ravel()
    rows.append({
        "model": name,
        "threshold_val": threshold,
        "AP": average_precision_score(actual, probabilities),
        "precision": precision_score(actual, labels, zero_division=0),
        "recall": recall_score(actual, labels, zero_division=0),
        "f1": f1_score(actual, labels, zero_division=0),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "test_n": len(actual),
    })

metrics = pd.DataFrame(rows).sort_values("AP", ascending=False)
display(metrics)

OUT_TABLES = ROOT / "reports" / "tables"
OUT_FIGURES = ROOT / "reports" / "figures"
OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_FIGURES.mkdir(parents=True, exist_ok=True)

metrics.to_csv(OUT_TABLES / "model_metrics.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 5.5))
prevalence = actual.mean()
label_map = {"dummy": "Dummy (Prior)", "rf": "Random Forest", "mlp": "MLP"}

for name in ["mlp", "rf", "dummy"]:
    probs = pred["p_" + name].to_numpy()
    ap = average_precision_score(actual, probs)
    prec, rec, _ = precision_recall_curve(actual, probs)
    ax.plot(rec, prec, lw=2, label=f"{label_map[name]} (AP = {ap:.4f})")

ax.axhline(prevalence, color="gray", linestyle="--", lw=1.2,
           label=f"Tỷ lệ trễ nền (Prevalence = {prevalence:.2%})")
ax.set_xlabel("Recall (Độ phủ đơn trễ)")
ax.set_ylabel("Precision (Độ chính xác cảnh báo)")
ax.set_title(f"Precision-Recall Curve trên tập Test (n = {len(actual):,})")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(OUT_FIGURES / "pr_curve_test.png", dpi=200)
plt.show()

## 5.2 Phân tích lỗi theo phân khúc

Cần chỉ ra khu vực, giá trị đơn, người bán/nhóm hàng có những cảnh báo sai hoặc bị bỏ sót. Chỉ phân tích slice có **đủ số mẫu**, tránh rút kết luận mạnh từ 2–3 đơn.

Có thể nối Test predictions với file `orders_model_ready.csv` bằng `order_id` (validate one_to_one), nhưng không coi các cột hậu nghiệm thành input của model.

In [ ]:
ready_file = PROCESSED / "orders_model_ready.csv"
if not ready_file.exists():
    raise FileNotFoundError("Thiếu orders_model_ready.csv từ Chương 3 để phân tích lỗi theo nhóm.")

orders_ready = pd.read_csv(ready_file)

print("Tổng số cột:", len(orders_ready.columns))

print("\nCác cột liên quan đến trọng lượng:")
weight_columns = [
    c for c in orders_ready.columns
    if "weight" in c.lower() or "peso" in c.lower()
]

print(weight_columns)

print("\nCác cột liên quan đến seller:")
print([
    c for c in orders_ready.columns
    if "seller" in c.lower()
])

print("\norder_id có duy nhất không?")
print(orders_ready["order_id"].is_unique)
champion_name = meta["champion"]
champ_thresh = float(meta["thresholds_from_val"][champion_name])


assert pred["order_id"].is_unique, "Test có order_id bị trùng!"
assert orders_ready["order_id"].is_unique, "Model-ready có order_id bị trùng!"

err_df = pred.merge(
    orders_ready,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

assert len(err_df) == len(pred), "Số dòng thay đổi sau khi nối!"

assert err_df["_merge"].eq("both").all(), (
    "Có đơn Test không tìm thấy trong orders_model_ready.csv!"
)

err_df = err_df.drop(columns=["_merge"])

print("PASS: Phép nối dữ liệu hợp lệ.")
print("Số đơn Test:", len(pred))
print("Số đơn sau khi nối:", len(err_df))


err_df["y_pred"] = (err_df["p_" + champion_name] >= champ_thresh).astype(int)
err_df["is_TP"] = ((err_df["y_true"] == 1) & (err_df["y_pred"] == 1)).astype(int)
err_df["is_FP"] = ((err_df["y_true"] == 0) & (err_df["y_pred"] == 1)).astype(int)
err_df["is_FN"] = ((err_df["y_true"] == 1) & (err_df["y_pred"] == 0)).astype(int)
err_df["is_TN"] = ((err_df["y_true"] == 0) & (err_df["y_pred"] == 0)).astype(int)

selected_metrics = (
    metrics.set_index("model").loc[champion_name]
)

for metric_name in ["TP", "FP", "FN", "TN"]:
    calculated = int(err_df["is_" + metric_name].sum())
    expected = int(selected_metrics[metric_name])

    assert calculated == expected, (
        f"{metric_name} không khớp: "
        f"{calculated} != {expected}"
    )

print("PASS: TP, FP, FN, TN khớp bảng tổng hợp.")

if "is_late" in err_df.columns:
    original_label = pd.to_numeric(
        err_df["is_late"],
        errors="coerce"
    )

    mismatch = (
        original_label.isna() |
        original_label.ne(err_df["y_true"])
    ).sum()

    print("Số nhãn không khớp:", mismatch)

prob_col = "p_" + champion_name

state_check = (
    err_df[err_df["customer_state"].isin(["SP", "RJ"])]
    .groupby("customer_state")
    .agg(
        n=("order_id", "count"),
        actual_late=("y_true", "sum"),
        min_prob=(prob_col, "min"),
        median_prob=(prob_col, "median"),
        max_prob=(prob_col, "max"),
        predicted_late=("y_pred", "sum"),
        TP=("is_TP", "sum"),
        FP=("is_FP", "sum"),
        FN=("is_FN", "sum")
    )
)

display(state_check)


group_cols = []
if "customer_state" in err_df.columns:
    group_cols.append("customer_state")

val_col = next((c for c in ["total_payment_value", "order_total_value", "price_sum", "total_price"] if c in err_df.columns), None)
if val_col:
    err_df["order_value_group"] = pd.qcut(err_df[val_col], q=4, duplicates="drop").astype(str)
    group_cols.append("order_value_group")


weight_candidates = [
    "total_weight_g",
    "product_weight_g_sum",
    "max_weight_g"
]

weight_col = next(
    (c for c in weight_candidates if c in err_df.columns),
    None
)

if weight_col is None:

    items_file = RAW / "olist_order_items_dataset.csv"
    products_file = RAW / "olist_products_dataset.csv"

    if items_file.exists() and products_file.exists():

        items = pd.read_csv(
            items_file,
            usecols=["order_id", "product_id"]
        )

        products = pd.read_csv(
            products_file,
            usecols=["product_id", "product_weight_g"]
        )

        assert products["product_id"].is_unique, (
            "product_id bị trùng trong bảng products!"
        )

        items_weight = items.merge(
            products,
            on="product_id",
            how="left",
            validate="many_to_one"
        )

        items_weight["product_weight_g"] = pd.to_numeric(
            items_weight["product_weight_g"],
            errors="coerce"
        )

        
        order_weight = (
            items_weight.groupby("order_id")["product_weight_g"]
            .agg(lambda s: s.sum(min_count=len(s)))
            .rename("total_weight_g")
            .reset_index()
        )

        err_df = err_df.merge(
            order_weight,
            on="order_id",
            how="left",
            validate="one_to_one"
        )

        weight_col = "total_weight_g"

    else:
        print("Thiếu dữ liệu raw để tính trọng lượng.")

if weight_col is not None:

    weight_values = pd.to_numeric(
        err_df[weight_col],
        errors="coerce"
    )

    valid_weight = (
        weight_values.notna() &
        (weight_values >= 0)
    )

    print("Cột trọng lượng sử dụng:", weight_col)
    print("Số đơn có trọng lượng hợp lệ:", valid_weight.sum())
    print("Số đơn thiếu/không hợp lệ:", (~valid_weight).sum())

    if (
        valid_weight.sum() >= 30
        and weight_values.loc[valid_weight].nunique() >= 2
    ):

        weight_bins = pd.qcut(
            weight_values.loc[valid_weight],
            q=4,
            duplicates="drop"
        )

        err_df["weight_group"] = "Thiếu/không hợp lệ"

        err_df.loc[valid_weight, "weight_group"] = (
            weight_bins.astype(str)
        )

        group_cols.append("weight_group")

        print("PASS: Đã tạo nhóm trọng lượng.")
        print("Số khoảng trọng lượng:",
              len(weight_bins.cat.categories))

    else:
        print("Chưa đủ dữ liệu để phân nhóm trọng lượng.")
else:
    print("Chưa thể phân tích trọng lượng vì thiếu dữ liệu.")


seller_col = next((c for c in ["n_sellers", "seller_count", "distinct_sellers"] if c in err_df.columns), None)
if seller_col:
    err_df["seller_group"] = np.where(err_df[seller_col] > 1, "Nhiều seller (>1)", "1 seller")
    group_cols.append("seller_group")

cat_col = next((c for c in ["main_category_en", "product_category_name_english", "top_category"] if c in err_df.columns), None)
if cat_col:
    group_cols.append(cat_col)

def summarize_error_by_group(df, col, min_n=30):
    g = df.groupby(col, dropna=False).agg(
        n=("order_id", "count"),
        actual_late=("y_true", "sum"),
        late_rate=("y_true", "mean"),
        TP=("is_TP", "sum"),
        FP=("is_FP", "sum"),
        FN=("is_FN", "sum"),
        TN=("is_TN", "sum"),
    ).reset_index()
    g = g[g["n"] >= min_n].copy()
    g["FPR"] = np.where((g["FP"] + g["TN"]) > 0, g["FP"] / (g["FP"] + g["TN"]), 0.0)
    g["precision"] = np.where((g["TP"] + g["FP"]) > 0, g["TP"] / (g["TP"] + g["FP"]), 0.0)
    g["FNR"] = np.where(
    (g["FN"] + g["TP"]) > 0,
    g["FN"] / (g["FN"] + g["TP"]),
    np.nan)

    g["recall"] = np.where(
    (g["TP"] + g["FN"]) > 0,
    g["TP"] / (g["TP"] + g["FN"]),
    np.nan)
    return g.sort_values(["late_rate", "n"], ascending=[False, False])

all_error_tables = []
for col in group_cols:
    tbl = summarize_error_by_group(err_df, col, min_n=30)
    print(f"\n=== Phân tích lỗi theo: {col} (n >= 30) ===")
    display(tbl.head(10))
    tbl_out = tbl.copy()
    tbl_out.insert(0, "dimension", col)
    tbl_out = tbl_out.rename(columns={col: "segment"})
    all_error_tables.append(tbl_out)

if all_error_tables:
    pd.concat(all_error_tables, ignore_index=True).to_csv(OUT_TABLES / "error_analysis_segments.csv", index=False)

state_tbl = summarize_error_by_group(err_df, "customer_state", min_n=30).sort_values("n", ascending=False).head(8)
fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(state_tbl))
w = 0.36
ax.bar(x - w/2, state_tbl["FPR"] * 100, width=w, label="FPR – Tỷ lệ cảnh báo sai (%)")
ax.bar(x + w/2, state_tbl["FNR"] * 100, width=w, label="FNR – Tỷ lệ bỏ sót đơn trễ (%)")
ax.set_xticks(x)
ax.set_xticklabels([f"{s}\n(n={n})" for s, n in zip(state_tbl["customer_state"], state_tbl["n"])])
ax.set_ylabel("Tỷ lệ lỗi (%)")
ax.set_title("Phân tích tỷ lệ cảnh báo sai (FPR) và bỏ sót (FNR) theo bang (Top 8 cỡ mẫu, n >= 30)")
ax.grid(axis="y", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(OUT_FIGURES / "error_by_state.png", dpi=200)
plt.show()

## 5.3 Đánh giá ý nghĩa vận hành: Top-K

Kịch bản: đội vận hành đủ khả năng theo dõi K đơn, cần chọn đơn có xác suất cao nhất. Tính **Precision@K**, **Recall@K**, **FP@K** và so sánh với cảnh báo theo ngưỡng.

K phải được chọn từ nhu cầu vận hành hoặc Validation; minh họa các giá trị K trên Test thì **chỉ để mô tả**, không dùng để tối ưu mô hình.

### 5.3.1 Phân tích xếp hạng Top-K theo ngân sách can thiệp vận hành
Đánh giá hiệu quả thực tế khi bộ phận vận hành chỉ có nguồn lực chủ động theo dõi hoặc cảnh báo sớm cho $K$ đơn hàng có xác suất giao trễ cao nhất ($K \in \{50, 100, 200, 5\%, 10\%, 20\%\}$ tổng số đơn tập Test), thông qua các chỉ số `Precision@K`, `Recall@K` và `Lift@K` so với tỷ lệ trễ nền.

In [ ]:
total_late = int(actual.sum())
n_test = len(actual)
k_list = [50, 100, 200, int(round(0.05 * n_test)), int(round(0.10 * n_test)), int(round(0.20 * n_test))]
k_list = sorted(list({k for k in k_list if 0 < k <= n_test}))

topk_rows = []
for name in ["mlp", "rf", "dummy"]:
    probs = pred["p_" + name].to_numpy()
    order_idx = np.argsort(-probs)
    y_sorted = actual[order_idx]
    for k in k_list:
        tp_k = int(y_sorted[:k].sum())
        fp_k = int(k - tp_k)
        prec_k = tp_k / k
        rec_k = tp_k / total_late if total_late > 0 else 0.0
        lift_k = prec_k / (total_late / n_test) if total_late > 0 else 0.0
        topk_rows.append({
            "model": name,
            "K": k,
            "pct_orders_checked": round(k / n_test * 100, 2),
            "TP_at_K": tp_k,
            "FP_at_K": fp_k,
            "Precision_at_K": prec_k,
            "Recall_at_K": rec_k,
            "Lift_at_K": lift_k,
            "total_late_test": total_late,
            "test_n": n_test,
        })

topk_df = pd.DataFrame(topk_rows)
display(topk_df[topk_df["model"] == champion_name])
topk_df.to_csv(OUT_TABLES / "topk_analysis.csv", index=False)

### 5.3.2 Kiểm tra tính toàn vẹn tệp đầu ra bàn giao
Xác nhận toàn bộ các bảng kết quả (`model_metrics.csv`, `topk_analysis.csv`, `error_analysis_segments.csv`) và biểu đồ (`pr_curve_test.png`) đã được xuất đầy đủ, đúng định dạng trước khi tổng hợp vào báo cáo Chương 5 và bàn giao cho Chương 6.

In [ ]:


expected_artifacts = [
    OUT_TABLES / "model_metrics.csv",
    OUT_TABLES / "topk_analysis.csv",
    OUT_TABLES / "error_analysis_segments.csv",
    OUT_FIGURES / "pr_curve_test.png",
    OUT_FIGURES / "error_by_state.png",
]

for p in expected_artifacts:
    assert p.exists(), f"Thiếu file đầu ra: {p}"
    assert p.stat().st_size > 0, f"File rỗng: {p}"
    print("OK:", p.relative_to(ROOT))

error_table = pd.read_csv(
    OUT_TABLES / "error_analysis_segments.csv"
)

required_dimensions = {
    "customer_state",
    "order_value_group",
    "weight_group",
    "seller_group",
}

actual_dimensions = set(
    error_table["dimension"].dropna()
)

missing_dimensions = (
    required_dimensions - actual_dimensions
)

assert not missing_dimensions, (
    f"Thiếu phân tích lỗi theo nhóm: {missing_dimensions}"
)

# Kiểm tra đủ ba mô hình
metrics_check = pd.read_csv(
    OUT_TABLES / "model_metrics.csv"
)

assert set(metrics_check["model"]) == {
    "dummy", "rf", "mlp"
}

assert metrics_check["test_n"].eq(len(pred)).all()

print("\nPASS: Các tệp và nhóm phân tích bắt buộc đã có.")


## 5.4 Thảo luận – so sánh mô hình & hạn chế

Cần trả lời:
- RF vs MLP khác nhau thế nào về AP, Precision, Recall, F1 và FP? Chênh lệch có đáng kể trong thực tiễn không?
- Số cảnh báo sai thay đổi như thế nào khi ưu tiên Top-K?
- Model mắc lỗi nhiều ở nhóm nào? Có thể do ít dữ liệu hoặc đặc trưng không đầy đủ?
- Những hạn chế: **delivered-only selection bias**, dữ liệu cũ (2016–2018), địa lý Brazil, không tự áp dụng kết quả cho sàn Việt Nam 2026, feature availability chưa xác minh hết.

**Không diễn giải AP là tỷ lệ phần trăm đơn đúng.** AP đo chất lượng xếp hạng Precision–Recall.

## 5.5 Checklist bàn giao
- [ ] Metrics Test cho Dummy/RF/MLP, có hỗ trợ số mẫu và nhãn.
- [ ] Confusion matrix và PR Curve từ dữ liệu thật.
- [ ] FP, FN, Precision@K, Recall@K và phân tích trade-off.
- [ ] Có bảng lỗi theo nhóm đủ kích thước mẫu, nhận xét trung lập.
- [ ] Có liên kết repo/commit của file dự đoán và ngưỡng Validation.
- [ ] Hoàn thành `reports/chapters/CH05_RESULTS.md`, gửi bảng/hình cho Tấn Lên.

**Cách trình bày:** cho thấy bảng kết quả trước, giải thích chỉ số theo góc nhìn vận hành, rồi phân tích sai và hạn chế; tránh chỉ khen mô hình có AP cao nhất.